# CÂU 2 – Bảng điều khiển phân tích biến động giá cổ phiếu (`abc-20.csv`)

**Mục tiêu:** xây dựng dashboard tương tác bằng `ipywidgets` để khám phá dữ liệu giá cổ phiếu, gồm 3 module theo đề:

| Module | Nội dung |
|---|---|
| ① Bộ lọc & nút điều khiển | chọn chuỗi giá, khoảng phiên, loại biểu đồ, MA, Bollinger, số dòng bảng; nút *Cập nhật / Đặt lại / Xuất CSV* |
| ② Bảng số liệu | thống kê tóm tắt + các dòng cuối của khoảng đã lọc |
| ③ Biểu đồ | giá + MA, so sánh 7 cột, phân phối lợi suất, biến động trượt |

**Đặc điểm của file dữ liệu (đã kiểm tra trước khi viết code):**
- **Không có dòng tiêu đề** → phải đọc bằng `header=None`, nếu đọc mặc định sẽ **mất phiên đầu tiên**.
- **365 phiên × 7 cột giá**, không có cột ngày, không có Open/High/Low/Volume.
- Vì không có ngày nên trục thời gian là **số thứ tự phiên** (không tự bịa ngày tháng).

**Cách chạy:** đặt `abc-20.csv` cùng thư mục với notebook → `Run All`. Cần: `pandas, numpy, matplotlib, seaborn, scipy, ipywidgets`.

## Bước 1 – Thư viện và giao diện đồ thị

In [ ]:
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

%matplotlib inline

# Giao diện chung cho mọi biểu đồ: nền sáng, lưới nhạt, tiêu đề đậm
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 100,
})

# Bảng màu dùng thống nhất trong toàn bộ dashboard
PRIMARY, UP, DOWN, ACCENT = "#2563eb", "#16a34a", "#dc2626", "#f59e0b"

## Bước 2 – Đọc và kiểm tra dữ liệu

Hai điểm mấu chốt:
1. `header=None` để giữ nguyên phiên đầu tiên.
2. Thêm cột **"Trung bình 7 cột"** để có một chuỗi đại diện, vì 7 cột đều là giá của cùng một mã.

In [ ]:
FILE_PATH = "abc-20.csv"

# header=None: file không có dòng tiêu đề
raw = pd.read_csv(FILE_PATH, header=None)
raw.columns = [f"Giá {i}" for i in range(1, raw.shape[1] + 1)]
raw.index = np.arange(1, len(raw) + 1)      # trục thời gian = số thứ tự phiên (1..365)
raw.index.name = "Phiên"

# df = 7 cột gốc + chuỗi trung bình (chuỗi đại diện)
df = raw.copy()
df["Trung bình 7 cột"] = raw.mean(axis=1)

SERIES = list(df.columns)       # danh sách chuỗi để đưa vào menu chọn
N = len(df)

print(f"Kích thước dữ liệu gốc : {raw.shape[0]} phiên × {raw.shape[1]} cột")
print(f"Số ô bị thiếu (NaN)    : {int(raw.isna().sum().sum())}")
print(f"Kiểu dữ liệu           : {raw.dtypes.unique()[0]}")
print("→ Dữ liệu sạch, không cần xử lý giá trị thiếu.\n")
display(raw.head())

## Bước 3 – Các hàm tính toán (chưa liên quan đến widget)

Tách phần **tính toán** khỏi phần **giao diện** giúp code dễ đọc và dễ kiểm tra.

> **Lưu ý quan trọng:** MA, Bollinger, biến động trượt được tính trên **toàn bộ 365 phiên**, rồi mới cắt theo khoảng phiên người dùng chọn.
> Nếu cắt trước rồi mới tính thì đầu khoảng sẽ bị trống (NaN) trong `window` phiên đầu.

In [ ]:
def lay_du_lieu(ten_cot, p_dau, p_cuoi, ma_ngan, ma_dai, cua_so_bd):
    """Tính các chỉ báo cho một chuỗi giá rồi cắt theo khoảng phiên [p_dau, p_cuoi]."""
    gia = df[ten_cot]
    out = pd.DataFrame({"Giá": gia})

    out["Lợi suất %"] = gia.pct_change() * 100                      # % thay đổi so với phiên trước
    out["MA ngắn"] = gia.rolling(ma_ngan).mean()                    # trung bình trượt ngắn hạn
    out["MA dài"] = gia.rolling(ma_dai).mean()                      # trung bình trượt dài hạn
    do_lech = gia.rolling(ma_ngan).std()                            # Bollinger = MA ± 2 độ lệch chuẩn
    out["BB trên"] = out["MA ngắn"] + 2 * do_lech
    out["BB dưới"] = out["MA ngắn"] - 2 * do_lech
    out["Biến động trượt %"] = out["Lợi suất %"].rolling(cua_so_bd).std()

    sub = out.loc[p_dau:p_cuoi].copy()
    # Drawdown = mức giảm so với đỉnh gần nhất TRONG khoảng đang xem
    sub["Drawdown %"] = (sub["Giá"] / sub["Giá"].cummax() - 1) * 100
    return sub


def thong_ke_tom_tat(sub):
    """Bảng thống kê tóm tắt của khoảng đã lọc (trả về DataFrame 1 cột để hiển thị)."""
    gia, ls = sub["Giá"], sub["Lợi suất %"].dropna()
    q1, q2, q3 = gia.quantile([0.25, 0.5, 0.75])
    dong = [
        ("Số phiên", f"{len(gia):,}"),
        ("Giá đầu kỳ", f"{gia.iloc[0]:,.2f}"),
        ("Giá cuối kỳ", f"{gia.iloc[-1]:,.2f}"),
        ("Thay đổi cả kỳ", f"{(gia.iloc[-1] / gia.iloc[0] - 1) * 100:+.2f} %"),
        ("Giá trung bình", f"{gia.mean():,.2f}"),
        ("Độ lệch chuẩn giá", f"{gia.std():,.2f}"),
        ("Giá thấp nhất", f"{gia.min():,.2f}  (phiên {gia.idxmin()})"),
        ("Giá cao nhất", f"{gia.max():,.2f}  (phiên {gia.idxmax()})"),
        ("Q1 / Trung vị / Q3", f"{q1:,.0f} / {q2:,.0f} / {q3:,.0f}"),
        ("Lợi suất TB mỗi phiên", f"{ls.mean():+.3f} %"),
        ("Độ lệch chuẩn lợi suất", f"{ls.std():.3f} %"),
        ("Độ lệch (skewness)", f"{ls.skew():+.3f}"),
        ("Độ nhọn (excess kurtosis)", f"{ls.kurt():+.3f}"),
        ("VaR 5% (1 phiên)", f"{np.percentile(ls, 5):.2f} %"),
        ("Drawdown tối đa", f"{sub['Drawdown %'].min():.2f} %"),
    ]
    return pd.DataFrame(dong, columns=["Chỉ tiêu", "Giá trị"]).set_index("Chỉ tiêu")


def bang_dong_cuoi(sub, n_dong, ma_ngan, ma_dai, hien_bb):
    """Bảng n dòng cuối của khoảng đã lọc."""
    cot = ["Giá", "Lợi suất %", "MA ngắn", "MA dài"] + (["BB trên", "BB dưới"] if hien_bb else [])
    t = sub[cot].tail(n_dong).rename(columns={"MA ngắn": f"MA{ma_ngan}", "MA dài": f"MA{ma_dai}"})
    return t

## Bước 4 – Các hàm vẽ biểu đồ (Module ③)

Mỗi hàm vẽ **một biểu đồ** và trả về `fig`. Dashboard sẽ gọi lần lượt 4 hàm này.

In [ ]:
def ve_bieu_do_gia(sub, ten, loai, hien_ma, hien_bb, ma_ngan, ma_dai):
    """Biểu đồ 1: giá + MA + Bollinger (3 kiểu: Đường / Diện tích / Cột)."""
    x, gia = sub.index, sub["Giá"]
    fig, ax = plt.subplots(figsize=(12, 5.5))
    handles = []

    if loai == "Cột":
        # Cột xanh = phiên tăng, đỏ = phiên giảm so với phiên trước
        mau = np.where(sub["Lợi suất %"].fillna(0) >= 0, UP, DOWN)
        ax.bar(x, gia, color=mau, width=0.9, alpha=0.75)
        handles += [plt.Rectangle((0, 0), 1, 1, color=UP, alpha=0.75, label="Phiên tăng"),
                    plt.Rectangle((0, 0), 1, 1, color=DOWN, alpha=0.75, label="Phiên giảm")]
    else:
        ax.plot(x, gia, color=PRIMARY, lw=1.8, label=f"Giá ({ten})")
        if loai == "Diện tích":
            ax.fill_between(x, gia, gia.min(), color=PRIMARY, alpha=0.15)

    if hien_bb:       # dải Bollinger: vùng mờ giữa BB trên và BB dưới
        ax.fill_between(x, sub["BB dưới"], sub["BB trên"], color="gray", alpha=0.15, label="Bollinger (±2σ)")
    if hien_ma:
        ax.plot(x, sub["MA ngắn"], color=ACCENT, lw=1.8, label=f"MA{ma_ngan}")
        ax.plot(x, sub["MA dài"], color="#7c3aed", lw=1.8, label=f"MA{ma_dai}")

    # Đánh dấu đỉnh và đáy của khoảng đang xem
    i_max, i_min = gia.idxmax(), gia.idxmin()
    ax.scatter([i_max], [gia[i_max]], color=UP, s=70, zorder=5, edgecolor="white")
    ax.scatter([i_min], [gia[i_min]], color=DOWN, s=70, zorder=5, edgecolor="white")
    def dat_nhan(i, nhan, mau, dy):
        # Điểm nằm nửa phải biểu đồ thì đặt chữ lệch sang trái để không bị cắt mép
        ben_phai = (i - x[0]) / max(x[-1] - x[0], 1) > 0.6
        ax.annotate(nhan, (i, gia[i]), xytext=(-8 if ben_phai else 8, dy), textcoords="offset points",
                    ha="right" if ben_phai else "left", fontsize=9, color=mau, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
    dat_nhan(i_max, f"Cao nhất {gia[i_max]:,.0f} (phiên {i_max})", UP, 8)
    dat_nhan(i_min, f"Thấp nhất {gia[i_min]:,.0f} (phiên {i_min})", DOWN, -16)

    h, _ = ax.get_legend_handles_labels()
    ax.legend(handles=h + handles, loc="upper center", bbox_to_anchor=(0.5, -0.12),
              ncol=6, frameon=False, fontsize=9)
    ax.set(title=f"{ten} — phiên {x[0]} đến {x[-1]}", xlabel="Số thứ tự phiên", ylabel="Giá")
    ax.set_xlim(x[0], x[-1])
    fig.tight_layout()
    return fig


def ve_so_sanh_7_cot(p_dau, p_cuoi):
    """Biểu đồ 2: so sánh 7 cột giá + ma trận tương quan lợi suất."""
    seg = raw.loc[p_dau:p_cuoi]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1.6, 1]})

    for mau, cot in zip(sns.color_palette("tab10", 7), seg.columns):
        ax1.plot(seg.index, seg[cot], color=mau, lw=1, alpha=0.6, label=cot)
    ax1.plot(seg.index, seg.mean(axis=1), color="black", lw=2.5, label="Trung bình")
    ax1.set(title="Giá của 7 cột và đường trung bình", xlabel="Số thứ tự phiên", ylabel="Giá")
    ax1.legend(ncol=4, fontsize=8, loc="upper left")
    ax1.set_xlim(seg.index[0], seg.index[-1])

    # Tương quan của LỢI SUẤT (không phải giá) mới phản ánh đúng mức "đi cùng nhau"
    ls = raw.pct_change().loc[p_dau:p_cuoi]
    corr = ls.corr()
    tb_corr = corr.values[np.triu_indices(7, 1)].mean()
    corr.index = corr.columns = [f"G{i}" for i in range(1, 8)]
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax2)
    ax2.set_title(f"Tương quan lợi suất (TB = {tb_corr:.2f})")
    fig.tight_layout()
    return fig


def ve_phan_phoi_loi_suat(sub, ten):
    """Biểu đồ 3: histogram + KDE + đường chuẩn lý thuyết, kèm Q-Q plot."""
    ls = sub["Lợi suất %"].dropna()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.5, 1]})

    sns.histplot(ls, bins=30, stat="density", color=PRIMARY, alpha=0.35, ax=ax1, label="Lợi suất thực tế")
    sns.kdeplot(ls, color=PRIMARY, lw=2, ax=ax1, label="KDE")
    xs = np.linspace(ls.min(), ls.max(), 300)
    ax1.plot(xs, stats.norm.pdf(xs, ls.mean(), ls.std()), color=ACCENT, ls="--", lw=2, label="Phân phối chuẩn")
    var5 = np.percentile(ls, 5)
    ax1.axvline(var5, color=DOWN, ls=":", lw=2, label=f"VaR 5% = {var5:.1f}%")
    ax1.axvline(ls.mean(), color="black", ls="-", lw=1, label=f"Trung bình = {ls.mean():.2f}%")

    p_sw = stats.shapiro(ls).pvalue
    ghi_chu = (f"n = {len(ls)}\nĐộ lệch chuẩn = {ls.std():.2f}%\nSkewness = {ls.skew():+.2f}\n"
               f"Kurtosis = {ls.kurt():+.2f}\nShapiro p = {p_sw:.3g}")
    ax1.text(0.98, 0.97, ghi_chu, transform=ax1.transAxes, ha="right", va="top", fontsize=9,
             bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cbd5e1"))
    ax1.set(title=f"Phân phối lợi suất mỗi phiên — {ten}", xlabel="Lợi suất (%)", ylabel="Mật độ")
    ax1.legend(loc="upper left", fontsize=9)

    # Q-Q plot: các điểm càng bám sát đường thẳng thì càng gần phân phối chuẩn
    stats.probplot(ls, dist="norm", plot=ax2)
    pts, line = ax2.get_lines()[0], ax2.get_lines()[1]
    pts.set(markerfacecolor=PRIMARY, markeredgecolor=PRIMARY, markersize=4, alpha=0.6)
    line.set(color=ACCENT, lw=2)
    ax2.set(title="Q-Q plot so với phân phối chuẩn", xlabel="Phân vị lý thuyết", ylabel="Phân vị mẫu")
    fig.tight_layout()
    return fig


def ve_bien_dong_truot(sub, ten, cua_so_bd):
    """Biểu đồ 4: độ biến động trượt (độ lệch chuẩn lợi suất) và drawdown."""
    x, bd = sub.index, sub["Biến động trượt %"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6.8), sharex=True,
                                   gridspec_kw={"height_ratios": [1.4, 1]})

    ax1.plot(x, bd, color=PRIMARY, lw=1.8)
    ax1.fill_between(x, bd, color=PRIMARY, alpha=0.15)
    ax1.axhline(bd.mean(), color=ACCENT, ls="--", lw=1.5, label=f"Trung bình = {bd.mean():.2f}%")
    i_cao = bd.idxmax()
    ax1.scatter([i_cao], [bd[i_cao]], color=DOWN, s=60, zorder=5, edgecolor="white")
    ax1.annotate(f"Biến động cao nhất {bd[i_cao]:.2f}% (phiên {i_cao})", (i_cao, bd[i_cao]),
                 xytext=(8, 6), textcoords="offset points", fontsize=9, color=DOWN, fontweight="bold")
    ax1.set(title=f"Biến động trượt {cua_so_bd} phiên — {ten}", ylabel="Độ lệch chuẩn lợi suất (%)")
    ax1.legend(loc="upper left", fontsize=9)

    ax2.fill_between(x, sub["Drawdown %"], 0, color=DOWN, alpha=0.35)
    ax2.plot(x, sub["Drawdown %"], color=DOWN, lw=1.2)
    ax2.set(title="Drawdown (mức giảm so với đỉnh gần nhất)", xlabel="Số thứ tự phiên", ylabel="Drawdown (%)")
    ax2.set_xlim(x[0], x[-1])
    fig.tight_layout()
    return fig

## Bước 5 – Module ①: Bộ lọc và nút điều khiển

Mỗi widget tương ứng một yêu cầu của đề. Dashboard **chỉ vẽ lại khi bấm "Cập nhật"** (để người dùng chỉnh nhiều tham số rồi mới chạy một lần).

In [ ]:
KIEU = {"description_width": "140px"}
RONG = widgets.Layout(width="440px")

# ---- Nhóm lọc dữ liệu
w_chuoi = widgets.Dropdown(options=SERIES, value="Trung bình 7 cột",
                           description="Chuỗi giá:", style=KIEU, layout=RONG)
w_khoang = widgets.IntRangeSlider(value=[1, N], min=1, max=N, step=1, description="Khoảng phiên:",
                                  continuous_update=False, style=KIEU, layout=RONG)
w_loai = widgets.ToggleButtons(options=["Đường", "Diện tích", "Cột"], value="Đường",
                               description="Loại biểu đồ:", style={"description_width": "140px", "button_width": "90px"})

# ---- Nhóm chỉ báo kỹ thuật
w_ma = widgets.Checkbox(value=True, description="Hiện đường MA", indent=False)
w_ma_ngan = widgets.IntSlider(value=20, min=5, max=60, description="MA ngắn (phiên):", style=KIEU, layout=RONG)
w_ma_dai = widgets.IntSlider(value=50, min=20, max=120, description="MA dài (phiên):", style=KIEU, layout=RONG)
w_bb = widgets.Checkbox(value=False, description="Hiện Bollinger (MA ngắn ± 2σ)", indent=False)
w_bd = widgets.IntSlider(value=20, min=5, max=60, description="Cửa sổ biến động:", style=KIEU, layout=RONG)
w_dong = widgets.IntSlider(value=10, min=5, max=50, step=5, description="Số dòng bảng:", style=KIEU, layout=RONG)

# ---- Nút điều khiển
btn_cap_nhat = widgets.Button(description="Cập nhật", button_style="primary", icon="refresh")
btn_dat_lai = widgets.Button(description="Đặt lại", button_style="warning", icon="undo")
btn_xuat = widgets.Button(description="Xuất CSV", button_style="success", icon="download")

# Giá trị mặc định (dùng cho nút Đặt lại)
MAC_DINH = [(w_chuoi, "Trung bình 7 cột"), (w_khoang, (1, N)), (w_loai, "Đường"),
            (w_ma, True), (w_ma_ngan, 20), (w_ma_dai, 50), (w_bb, False), (w_bd, 20), (w_dong, 10)]

## Bước 6 – Gắn logic cho nút bấm và ráp giao diện

- **Cập nhật:** kiểm tra tham số → tính chỉ báo → hiển thị bảng (Module ②) và 4 biểu đồ (Module ③).
- **Đặt lại:** trả mọi widget về mặc định rồi cập nhật.
- **Xuất CSV:** tạo đường link tải file `abc20_loc_<đầu>_<cuối>.csv` gồm khoảng dữ liệu đang xem kèm các chỉ báo (link nhúng sẵn dữ liệu nên dùng được cả khi chạy trên web).

In [ ]:
# Các vùng hiển thị kết quả
out_thong_bao = widgets.Output()
out_bang = widgets.Output()
TEN_TAB = ["Giá + MA", "So sánh 7 cột", "Phân phối lợi suất", "Biến động trượt"]
out_tabs = [widgets.Output() for _ in TEN_TAB]
tab = widgets.Tab(children=out_tabs)
for i, t in enumerate(TEN_TAB):
    tab.set_title(i, t)

TRANG_THAI = {"sub": None, "p_dau": None, "p_cuoi": None}    # nhớ dữ liệu lần cập nhật gần nhất


def cap_nhat(_=None):
    ten, (p_dau, p_cuoi) = w_chuoi.value, w_khoang.value
    ma_n, ma_d, cs_bd, n_dong = w_ma_ngan.value, w_ma_dai.value, w_bd.value, w_dong.value

    # --- kiểm tra tham số hợp lệ
    with out_thong_bao:
        clear_output(wait=True)
        if ma_d <= ma_n:
            print("⚠️ MA dài phải lớn hơn MA ngắn. Hãy chỉnh lại rồi bấm Cập nhật.")
            return
        if p_cuoi - p_dau < 10:
            print("⚠️ Hãy chọn khoảng ít nhất 10 phiên để thống kê có ý nghĩa.")
            return

    sub = lay_du_lieu(ten, p_dau, p_cuoi, ma_n, ma_d, cs_bd)
    TRANG_THAI.update(sub=sub, p_dau=p_dau, p_cuoi=p_cuoi)

    # --- Module ②: bảng số liệu
    with out_bang:
        clear_output(wait=True)
        display(HTML(f"<h4>Thống kê tóm tắt — {ten}, phiên {p_dau} → {p_cuoi}</h4>"))
        display(thong_ke_tom_tat(sub))
        display(HTML(f"<h4>{n_dong} phiên cuối của khoảng đã lọc</h4>"))
        bang = bang_dong_cuoi(sub, n_dong, ma_n, ma_d, w_bb.value)
        try:    # tô màu lợi suất tăng/giảm nếu môi trường hỗ trợ Styler
            sty = bang.style.format("{:,.2f}", na_rep="–")
            tomau = sty.map if hasattr(sty, "map") else sty.applymap
            display(tomau(lambda v: f"color: {UP}" if v > 0 else (f"color: {DOWN}" if v < 0 else ""),
                          subset=["Lợi suất %"]))
        except Exception:
            display(bang.round(2))

    # --- Module ③: 4 biểu đồ
    with out_tabs[0]:
        clear_output(wait=True)
        ve_bieu_do_gia(sub, ten, w_loai.value, w_ma.value, w_bb.value, ma_n, ma_d)
        plt.show()
    with out_tabs[1]:
        clear_output(wait=True)
        ve_so_sanh_7_cot(p_dau, p_cuoi)
        plt.show()
    with out_tabs[2]:
        clear_output(wait=True)
        ve_phan_phoi_loi_suat(sub, ten)
        plt.show()
    with out_tabs[3]:
        clear_output(wait=True)
        ve_bien_dong_truot(sub, ten, cs_bd)
        plt.show()


def dat_lai(_=None):
    for w, v in MAC_DINH:
        w.value = v
    out_thong_bao.clear_output()
    cap_nhat()


def xuat_csv(_=None):
    if TRANG_THAI["sub"] is None:
        cap_nhat()
    if TRANG_THAI["sub"] is None:       # cập nhật thất bại do tham số không hợp lệ
        return
    sub = TRANG_THAI["sub"]
    ten_file = f"abc20_loc_{TRANG_THAI['p_dau']}_{TRANG_THAI['p_cuoi']}.csv"
    # Nhúng file CSV thẳng vào đường link (base64) -> tải được cả trên web (Voilà/Render),
    # không cần ghi file lên máy chủ và mỗi người dùng nhận đúng dữ liệu mình vừa lọc.
    b64 = base64.b64encode(sub.round(4).to_csv().encode("utf-8-sig")).decode()
    link = (f'<a download="{ten_file}" href="data:text/csv;base64,{b64}" '
            f'style="font-weight:bold;color:#2563eb">⬇️ Bấm để tải {ten_file}</a>')
    with out_thong_bao:
        clear_output(wait=True)
        print(f"✅ Đã chuẩn bị {len(sub)} dòng dữ liệu đã lọc.")
        display(HTML(link))


btn_cap_nhat.on_click(cap_nhat)
btn_dat_lai.on_click(dat_lai)
btn_xuat.on_click(xuat_csv)

## Bước 7 – Hiển thị dashboard

In [ ]:
def tieu_de(txt):
    return widgets.HTML(f"<h3 style='margin:12px 0 4px'>{txt}</h3>")

khung_loc = widgets.VBox([widgets.HTML("<b>🔎 Bộ lọc dữ liệu</b>"), w_chuoi, w_khoang, w_loai],
                         layout=widgets.Layout(border="1px solid #cbd5e1", padding="8px", margin="0 8px 0 0"))
khung_chi_bao = widgets.VBox([widgets.HTML("<b>📐 Chỉ báo & bảng</b>"), w_ma, w_ma_ngan, w_ma_dai,
                              w_bb, w_bd, w_dong],
                             layout=widgets.Layout(border="1px solid #cbd5e1", padding="8px"))

giao_dien = widgets.VBox([
    widgets.HTML("<h2>📈 Dashboard phân tích biến động giá cổ phiếu ABC-20</h2>"),
    tieu_de("① Bộ lọc & nút điều khiển"),
    widgets.HBox([khung_loc, khung_chi_bao]),
    widgets.HBox([btn_cap_nhat, btn_dat_lai, btn_xuat]),
    out_thong_bao,
    tieu_de("② Bảng số liệu"),
    out_bang,
    tieu_de("③ Biểu đồ"),
    tab,
])

display(giao_dien)
cap_nhat()      # vẽ lần đầu với tham số mặc định

## Bước 8 – Kiểm tra nhanh toàn bộ dữ liệu (để viết nhận xét)

Ô này chạy độc lập với dashboard, in ra các con số nền tảng cho phần nhận xét ngay bên dưới.

In [ ]:
ls_all = raw.pct_change().dropna() * 100
tb = df["Trung bình 7 cột"]
ls_tb = tb.pct_change().dropna() * 100
iu = np.triu_indices(raw.shape[1], 1)            # lấy các cặp cột khác nhau (bỏ đường chéo)

print("1) Tương quan giữa 7 cột")
print(f"   - Tương quan GIÁ     trung bình : {raw.corr().values[iu].mean():.2f}")
print(f"   - Tương quan LỢI SUẤT trung bình : {ls_all.corr().values[iu].mean():.2f}")
print("   → Giá tương quan rất cao vì cùng xu hướng, nhưng lợi suất chỉ tương quan yếu:")
print("     7 cột là 7 chuỗi khá độc lập về biến động, KHÔNG phải bản sao của nhau.\n")

print("2) Độ biến động mỗi phiên (độ lệch chuẩn lợi suất, %)")
print(ls_all.std().round(2).to_string())
print(f"   Chuỗi trung bình: {ls_tb.std():.2f}%")
print("   → Lấy trung bình 7 cột làm giảm gần một nửa biến động (hiệu ứng đa dạng hoá).\n")

print("3) Chuỗi trung bình trên toàn kỳ")
print(f"   - Giá đầu → cuối : {tb.iloc[0]:,.0f} → {tb.iloc[-1]:,.0f}  ({(tb.iloc[-1]/tb.iloc[0]-1)*100:+.1f}%)")
print(f"   - Cao nhất       : {tb.max():,.0f} (phiên {tb.idxmax()})")
print(f"   - Thấp nhất      : {tb.min():,.0f} (phiên {tb.idxmin()})")
print(f"   - Drawdown tối đa: {((tb/tb.cummax()-1).min())*100:.1f}%")
print(f"   - Skewness / Kurtosis lợi suất: {ls_tb.skew():+.2f} / {ls_tb.kurt():+.2f}")
print(f"\n4) Tỉ lệ phiên có |lợi suất| ≥ 10% theo từng cột (%)")
print((ls_all.abs().ge(9.99).mean() * 100).round(0).astype(int).to_string())

## Nhận xét và kết luận

1. **Về dữ liệu:** 365 phiên × 7 cột, không thiếu giá trị. Phải đọc với `header=None`, vì file không có dòng tiêu đề; đọc mặc định sẽ mất phiên đầu tiên.
2. **7 cột có phải "cùng một thị trường" không?** Giá của 7 cột tương quan rất cao (≈ 0,93–0,95) nhưng **lợi suất chỉ tương quan ≈ 0,23**. Tương quan giá cao chủ yếu do các chuỗi cùng có xu hướng; lấy sai phân (lợi suất) cho thấy biến động từng phiên khá độc lập. Riêng cột 7 gần như không tương quan với các cột còn lại (≈ 0,00–0,02 ở lợi suất). Vì vậy, dashboard cho phép xem từng cột **và** chuỗi trung bình để so sánh; nên thận trọng khi coi 7 cột là cùng một chuỗi.
3. **Biến động rất lớn:** độ lệch chuẩn lợi suất khoảng 14–16 %/phiên ở 6 cột đầu (cột 7 ≈ 8 %, vì không bao giờ vượt ±10 %), và hơn một nửa số phiên có |lợi suất| ≥ 10 %. Mức này **cao bất thường** so với cổ phiếu thực tế (thường 1–3 %/phiên), nên nhiều khả năng đây là **dữ liệu mô phỏng**; chỉ nên dùng để thực hành phân tích, không nên rút ra kết luận đầu tư.
4. **Chuỗi trung bình** ổn định hơn: biến động khoảng 8,6 %/phiên, giá đi từ ≈ 10 460 lên ≈ 14 044 (+34 %) nhưng dao động rất mạnh (đáy ≈ 3 979 ở phiên 172, đỉnh ≈ 21 595 ở phiên 286; drawdown tối đa ≈ −81 %).
5. **Phân phối lợi suất:** gần đối xứng (skewness ≈ 0) nhưng **dẹt hơn phân phối chuẩn** (excess kurtosis ≈ −1) và có xu hướng **hai đỉnh** (quanh −8 % và +5 %) — khác hẳn đặc điểm "đuôi dày" của thị trường thật, càng củng cố nhận định ở mục 3. Q-Q plot và kiểm định Shapiro–Wilk trong dashboard cho thấy lợi suất không tuân theo phân phối chuẩn.
6. **Hướng mở rộng:** thêm cột ngày nếu có dữ liệu thật; thêm RSI/MACD; hoặc triển khai lên web bằng **Voilà** (`voila bai_2_dashboard.ipynb`).